# 3DGS: full handwritten Metal render pipeline

This experimental notebook keeps PyTorch as the tensor owner and dispatch layer,
but moves the render front-end and rasterization to handwritten Metal kernels.

Pipeline:

```text
Gaussian setup / projection
→ rectangular screen-space AABB
→ hierarchical exclusive scan
→ Gaussian/tile intersection emission
→ stable 4-bit LSD radix sort by (tile_id, depth)
→ dense tile counts + exclusive scan
→ one Metal threadgroup per tile
→ front-to-back alpha compositing
```

The Metal source lives in `metal_kernels/`; orchestration lives in
`metal_renderer.py`. The PyTorch implementation is kept separately in
`pytorch_reference_renderer.py` only as a correctness/performance reference.

The only intentional GPU→CPU synchronization inside the Metal front-end is the
single scalar `K = number of Gaussian/tile intersections`, because PyTorch must
know `K` before allocating the variable-length intersection buffers.


In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch

from metal_renderer import (
    gaussian_rasterization_metal,
    get_last_metal_stats,
)
from pytorch_reference_renderer import gaussian_rasterization_pytorch
from util import build_covariance, load_cameras, scale_intrinsics

In [ ]:
scene = "bonsai"
device = torch.device("mps")

if not torch.backends.mps.is_available():
    raise RuntimeError("This notebook requires an Apple Silicon MPS device")

pos = torch.from_numpy(
    torch.load("out_bonsai/pos_param.pt", weights_only=False)
).to(device)
opacity_raw = torch.from_numpy(
    torch.load("out_bonsai/alpha_raw_param.pt", weights_only=False)
).to(device)
color = torch.sigmoid(
    0.282 * torch.from_numpy(
        torch.load("out_bonsai/f_dc.pt", weights_only=False)
    )
).to(device)
scale_raw = torch.from_numpy(
    torch.load("out_bonsai/scale_raw.pt", weights_only=False)
).to(device)
rot_raw = torch.from_numpy(
    torch.load("out_bonsai/rot_raw.pt", weights_only=False)
).to(device)

sigma = build_covariance(scale_raw, rot_raw)

cam_parameters = np.load(
    f"out_colmap/{scene}/cam_meta.npy",
    allow_pickle=True,
).item()

H_source = cam_parameters["height"]
W_source = cam_parameters["width"]
fx_source = cam_parameters["fx"]
fy_source = cam_parameters["fy"]
cx_source = W_source / 2
cy_source = H_source / 2

H = H_source // 2
W = W_source // 2
fx, fy, cx, cy = scale_intrinsics(
    H,
    W,
    H_source,
    W_source,
    fx_source,
    fy_source,
    cx_source,
    cy_source,
)

c2ws, image_paths = load_cameras(
    f"out_colmap/{scene}/cameras.npy",
    f"image_data/{scene}/images_2",
)

CAM_ID = 10
c2w = c2ws[CAM_ID].to(device)
image_path = image_paths[CAM_ID]

renderer_args = (
    pos,
    color,
    opacity_raw,
    sigma,
    c2w,
    H,
    W,
    fx,
    fy,
    cx,
    cy,
)

print(f"Gaussians: {pos.shape[0]:,}")
print(f"Image: {W} × {H}")
print(f"Camera: {CAM_ID}")

## Metal implementation notes

`gaussian_setup.metal` performs projection, frustum filtering, covariance
projection/stabilization, inverse conic construction and rectangular 3σ AABB
construction. Invisible Gaussians simply emit `tile_count = 0`, so no PyTorch
mask compaction is needed.

`scan.metal` provides a hierarchical exclusive scan. `binning.metal` emits the
variable-length Gaussian/tile records. `radix_sort.metal` uses stable 4-bit LSD
passes: depth bits first, then tile-id bits, yielding `(tile_id, depth)` order.

Finally `tile_rasterizer.metal` consumes dense tile offsets and performs
front-to-back alpha compositing with one threadgroup per image tile.


In [ ]:
def synchronize():
    torch.mps.synchronize()


def measure(renderer, *args):
    synchronize()
    started_at = perf_counter()
    image = renderer(*args)
    synchronize()
    return image, perf_counter() - started_at


# Compile all Metal libraries and warm allocator/caches before timing.
_ = gaussian_rasterization_metal(*renderer_args)
synchronize()

img_pytorch, pytorch_seconds = measure(
    gaussian_rasterization_pytorch,
    *renderer_args,
)
img_metal, metal_seconds = measure(
    gaussian_rasterization_metal,
    *renderer_args,
)

print(f"PyTorch reference: {pytorch_seconds * 1_000:.2f} ms")
print(f"Full Metal pipeline: {metal_seconds * 1_000:.2f} ms")
print(f"Speedup: {pytorch_seconds / metal_seconds:.2f}×")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(img_pytorch.detach().cpu().numpy())
axes[0].set_title("PyTorch reference")
axes[0].axis("off")

axes[1].imshow(img_metal.detach().cpu().numpy())
axes[1].set_title("Full Metal pipeline")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
stats = get_last_metal_stats()

print("Metal pipeline diagnostics")
for key, value in stats.items():
    if key == "total_seconds":
        print(f"  {key}: {value * 1_000:.2f} ms")
    else:
        print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

In [ ]:
absolute_difference = (img_pytorch - img_metal).abs()

strict_atol = 2e-3
sparse_outlier_atol = 5e-3
maximum_sparse_outlier_fraction = 1e-5

max_absolute_difference = absolute_difference.max().item()
mean_absolute_difference = absolute_difference.mean().item()
strict_mismatches = (absolute_difference > strict_atol).sum().item()
strict_mismatch_fraction = strict_mismatches / absolute_difference.numel()

print(f"Maximum absolute difference: {max_absolute_difference:.8f}")
print(f"Mean absolute difference: {mean_absolute_difference:.8f}")
print(f"Values with absolute difference > {strict_atol}: {strict_mismatches}")
print(f"Sparse mismatch fraction: {strict_mismatch_fraction:.10%}")

# MPSGraph/PyTorch and handwritten Metal can put a few values on
# opposite sides of floating-point boundaries. Allow a tiny number
# of sparse outliers while still rejecting systematic disagreement.
assert strict_mismatch_fraction <= maximum_sparse_outlier_fraction, (
    f"Too many values differ by more than {strict_atol}: "
    f"{strict_mismatches} / {absolute_difference.numel()}"
)

torch.testing.assert_close(
    img_metal,
    img_pytorch,
    rtol=1e-4,
    atol=sparse_outlier_atol,
)
print("Pixel-wise comparison passed with sparse float32 outliers allowed.")

plt.figure(figsize=(8, 6))
plt.imshow(absolute_difference.max(dim=-1).values.detach().cpu().numpy())
plt.title("Maximum absolute difference per pixel")
plt.colorbar()
plt.axis("off")
plt.show()
